In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Data Preprocessing and *Augmentation*

In [2]:
from torchvision import datasets, transforms

# Training images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Dataset Loading and Train/Validation Split

In [9]:
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy
import numpy as np

raw = Path("/content/drive/MyDrive/ASL_Project/asl_dataset/asl_dataset")

train_data = datasets.ImageFolder(raw, transform=train_transform)

indices = np.arange(len(train_data))
targets = np.array(train_data.targets)

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

print("Training images:", len(train_indices))
print("Validation images:", len(val_indices))

Training images: 2012
Validation images: 503


In [16]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
import torch.nn as nn
import torch
import time

# Dataset for validation without augmentation
val_data = datasets.ImageFolder(raw, transform=val_transform)

train_subset = Subset(train_data, train_indices)
val_subset = Subset(val_data, val_indices)

train_loader = DataLoader(
    train_subset, batch_size=32, shuffle=True
)

val_loader = DataLoader(
    val_subset, batch_size=32, shuffle=False
)

# Get class names from the dataset
class_names = train_data.classes

# Count the number of classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Class labels:", class_names)

Number of classes: 36
Class labels: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


ResNet18 Transfer Learning Model

In [19]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet18
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Device:", device)
print(model.fc)
print("Output classes:", model.fc.out_features)

Device: cuda
Linear(in_features=512, out_features=36, bias=True)
Output classes: 36


ResNet18 Model Train

In [20]:
import time

epochs = 3
start_time = time.time()

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss/total:.4f} "
        f"Accuracy: {100*correct/total:.2f}%"
    )

training_time = time.time() - start_time
print("Total training time:", round(training_time, 2), "seconds")

Epoch [1/3] Loss: 1.5124 Accuracy: 69.33%
Epoch [2/3] Loss: 0.2797 Accuracy: 95.03%
Epoch [3/3] Loss: 0.1245 Accuracy: 98.01%
Total training time: 68.26 seconds


Model Evaluation

In [21]:
model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Validation Loss:", round(val_loss / total, 4))
print("Validation Accuracy:", round(100 * correct / total, 2), "%")
print("Training Time:", round(training_time, 2), "seconds")

Validation Loss: 0.0953
Validation Accuracy: 96.82 %
Training Time: 68.26 seconds


Saving the model

In [22]:
from pathlib import Path
import torch

save_path = Path("/content/drive/MyDrive/ASL_Project/results")
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": class_names,
    "num_classes": num_classes
}, save_path / "restnet_18_asl.pth")

print("Model saved:", save_path / "restnet_18_asl.pth")
print("Class labels:", class_names)

Model saved: /content/drive/MyDrive/ASL_Project/results/restnet_18_asl.pth
Class labels: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
